# CURE-Rec — remaining review closure

This notebook is the single control surface for the remaining review items. It never edits the registered YAML configuration. Run all cells from top to bottom. Actions that require unavailable data or a new policy implementation are recorded as explicit skips; no result is fabricated.


In [1]:
from pathlib import Path
import sys,json,re,time
import pandas as pd

C=[Path.cwd(),Path.cwd()/'paper-ideas'/'CURE-Rec'/'code',*Path.cwd().parents]
ROOT=next(p for p in C if (p/'pyproject.toml').exists() and (p/'cure_rec').exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from cure_rec.config import load_settings
from cure_rec.pipeline import run_experiment
from cure_rec.revision_suite import objective_ablation, paired_user_statistics

CONFIG=ROOT/'configs'/'curesim_full.yaml'
RESULTS=ROOT/'results'/'reviewer_phase_assets'
PAPER=ROOT.parent/'paper'/'cure-rec-springer'/'cure-rec.tex'
RUN_ALL=True
RUN_DIVERGENT_SELECTOR=True
RUN_UTILITY_SENSITIVITY=True
RUN_CONSTRAINT_FRONTIER=True
RUN_SEMI_REAL=False  # requires a declared replay/world-model protocol
RUN_INTEGRATED_SCALING=False  # requires actual 8/10 policy operators
print('Root:',ROOT)

Root: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## 1. Manuscript consistency audit


In [2]:
text=PAPER.read_text()
checks={
 'free_F4_user_index': bool(re.search(r'Q_\{j,t\+1\}.*1\{j\\in L_\{u,t\}\}',text)),
 'old_decision_faithful': 'decision-faithful attribution' in text,
 'old_preregistered': 'preregistered' in text,
 'old_reviewer_language': 'reviewer-revision checks' in text or 'Reviewer-revision evidence' in text,
 'table_3_reference': 'Table 3' in text,
 'artifact_section': 'Artifact inventory' in text,
 'cost_definition': 'c(S)=\\sum' in text,
 'gini_definition': 'G(x)=' in text,
}
print(json.dumps(checks,indent=2))
if checks['old_decision_faithful'] or checks['old_preregistered'] or checks['old_reviewer_language'] or checks['free_F4_user_index']:
    print('Manuscript still has terminology/notation items requiring manual editing.')

{
  "free_F4_user_index": false,
  "old_decision_faithful": false,
  "old_preregistered": false,
  "old_reviewer_language": true,
  "table_3_reference": false,
  "artifact_section": true,
  "cost_definition": false,
  "gini_definition": true
}
Manuscript still has terminology/notation items requiring manual editing.


## 2. Utility-weight sensitivity

Runs predeclared alternative utility vectors on one exact full game each. Results are sensitivity evidence, not performance gains.

In [3]:
WEIGHTS=[
 {'name':'satisfaction_heavy','satisfaction_weight':0.85,'retention_weight':0.15,'fatigue_weight':0.35,'cost_weight':1.0},
 {'name':'retention_heavy','satisfaction_weight':0.35,'retention_weight':0.65,'fatigue_weight':0.35,'cost_weight':1.0},
 {'name':'fatigue_averse','satisfaction_weight':0.70,'retention_weight':0.30,'fatigue_weight':0.70,'cost_weight':1.0},
 {'name':'cost_averse','satisfaction_weight':0.70,'retention_weight':0.30,'fatigue_weight':0.35,'cost_weight':2.0},
]
if RUN_ALL and RUN_UTILITY_SENSITIVITY:
 rows=[]
 for w in WEIGHTS:
  cfg=load_settings(CONFIG); cfg.run.name='utility-'+w['name']; cfg.run.output_root=ROOT/'runs'/'reviewer-closure';
  for k,v in w.items():
   if k!='name': setattr(cfg.utility,k,v)
  logger,game,decision=run_experiment(cfg)
  rows.append({'setting':w['name'],'portfolio':';'.join(decision.selected_interventions),'status':decision.status.value,'mode':decision.mode.value,'lower_improvement':decision.lower_improvement,'base_feasible':decision.base_feasible,'source_run':str(logger.run_dir)})
 utility=pd.DataFrame(rows); out=RESULTS/'reviewer_closure'; out.mkdir(parents=True,exist_ok=True); utility.to_csv(out/'utility_weight_sensitivity.csv',index=False); display(utility)
else: print('Utility sensitivity skipped.')

2026-08-18 01:38:28,496 | INFO | run_started | {"config_hash": "787db96ce0a1ccb0", "run_id": "utility-satisfaction_heavy-20260818T003828Z-ec4ecb31"}
2026-08-18 01:38:28,497 | INFO | exact_game_started | {}
2026-08-18 01:38:28,498 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-18 01:41:06,288 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.1739479533216729, "scenario": "nominal", "shapley_efficiency_gap": -2.7755575615628914e-17}
2026-08-18 01:41:06,289 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-18 01:43:48,445 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.17480596369523932, "scenario": "fatigue_stress", "shapley_efficiency_gap": -5.551115123125783e-17}
2026-08-18 01:43:48,446 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-18 01:46:26,480 | INFO |

,setting,portfolio,status,mode,lower_improvement,base_feasible,source_run
0,satisfaction_heavy,repeat_cap,repair_selected,repair,0.257058,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
1,retention_heavy,repeat_cap,repair_selected,repair,0.381849,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
2,fatigue_averse,repeat_cap,repair_selected,repair,0.461201,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
3,cost_averse,repeat_cap,repair_selected,repair,0.244495,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...


## 3. Constraint frontier

Runs a predeclared grid of budget, relevance, provider and fatigue thresholds.


In [4]:
FRONTIER=[
 {'name':'strict_provider','budget':0.35,'relevance':-0.08,'provider':0.24,'fatigue':0.65},
 {'name':'baseline','budget':0.35,'relevance':-0.08,'provider':0.28,'fatigue':0.65},
 {'name':'relaxed_provider','budget':0.35,'relevance':-0.08,'provider':0.34,'fatigue':0.65},
 {'name':'tight_budget','budget':0.20,'relevance':-0.08,'provider':0.28,'fatigue':0.65},
 {'name':'tight_relevance','budget':0.35,'relevance':-0.03,'provider':0.28,'fatigue':0.65},
 {'name':'tight_fatigue','budget':0.35,'relevance':-0.08,'provider':0.28,'fatigue':0.45},
]
if RUN_ALL and RUN_CONSTRAINT_FRONTIER:
 rows=[]
 for f in FRONTIER:
  cfg=load_settings(CONFIG); cfg.run.name='frontier-'+f['name']; cfg.run.output_root=ROOT/'runs'/'reviewer-closure'; cfg.constraints.budget=f['budget']; cfg.constraints.min_relevance_delta=f['relevance']; cfg.constraints.max_provider_disparity=f['provider']; cfg.constraints.max_fatigue=f['fatigue']
  logger,game,decision=run_experiment(cfg); rows.append({'setting':f['name'],**f,'portfolio':';'.join(decision.selected_interventions),'status':decision.status.value,'mode':decision.mode.value,'lower_improvement':decision.lower_improvement,'base_feasible':decision.base_feasible,'source_run':str(logger.run_dir)})
 frontier=pd.DataFrame(rows); out=RESULTS/'reviewer_closure'; out.mkdir(parents=True,exist_ok=True); frontier.to_csv(out/'constraint_frontier.csv',index=False); display(frontier)
else: print('Constraint frontier skipped.')

2026-08-18 02:41:49,933 | INFO | run_started | {"config_hash": "c194e15c8164b388", "run_id": "frontier-strict_provider-20260818T014149Z-0d8d0944"}
2026-08-18 02:41:49,934 | INFO | exact_game_started | {}
2026-08-18 02:41:49,935 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-18 02:44:31,255 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13323832372399413, "scenario": "nominal", "shapley_efficiency_gap": 1.3877787807814457e-16}
2026-08-18 02:44:31,256 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-18 02:47:13,002 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13544545518030754, "scenario": "fatigue_stress", "shapley_efficiency_gap": 0.0}
2026-08-18 02:47:13,003 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-18 02:49:52,591 | INFO | scenario_game_comple

,setting,name,budget,relevance,provider,fatigue,portfolio,status,mode,lower_improvement,base_feasible,source_run
0,strict_provider,strict_provider,0.35,-0.08,0.24,0.65,repeat_cap;diversify,repair_selected,repair,0.234653,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
1,baseline,baseline,0.35,-0.08,0.28,0.65,repeat_cap,repair_selected,repair,0.294495,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
2,relaxed_provider,relaxed_provider,0.35,-0.08,0.34,0.65,repeat_cap,improve_selected,improvement,0.294495,True,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
3,tight_budget,tight_budget,0.20,-0.08,0.28,0.65,repeat_cap,repair_selected,repair,0.294495,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
4,tight_relevance,tight_relevance,0.35,-0.03,0.28,0.65,provider_balance,repair_selected,repair,-0.114157,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...
5,tight_fatigue,tight_fatigue,0.35,-0.08,0.28,0.45,repeat_cap,repair_selected,repair,0.294495,False,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...


## 4. Divergent selector protocol

The primary full configuration has selector ties. This cell records the mandatory protocol: select only on calibration seeds, freeze masks, and evaluate on disjoint seeds. A full implementation must use completed game tables from configurations where masks differ; it must not treat a single dominant repeat-cap configuration as evidence of selector superiority.

In [5]:
if RUN_ALL and RUN_DIVERGENT_SELECTOR:
 print('Divergent-selector action requires archived configurations with different masks; primary tied result is retained as a null comparison.')
 print('Required output columns: configuration, selector, selection_seed, evaluation_seed, frozen_mask, robust_regret, feasible, constraint margins.')
else: print('Divergent selector protocol skipped.')

Divergent-selector action requires archived configurations with different masks; primary tied result is retained as a null comparison.
Required output columns: configuration, selector, selection_seed, evaluation_seed, frozen_mask, robust_regret, feasible, constraint margins.


## 5. Semi-real intervention validation


In [6]:
if RUN_SEMI_REAL:
 raise NotImplementedError('Requires audited replay/world-model intervention semantics; no unsupported real intervention claim is generated.')
else: print('Semi-real intervention validation skipped: audited intervention log/world model unavailable.')

Semi-real intervention validation skipped: audited intervention log/world model unavailable.


## 6. Integrated 8/10-player scaling


In [7]:
if RUN_INTEGRATED_SCALING:
 raise NotImplementedError('Distinct 8/10-player policy operators must be integrated into CURE-Sim before this study can run.')
else: print('Integrated scaling skipped: current 8/10 result is arithmetic attribution stress testing only.')

Integrated scaling skipped: current 8/10 result is arithmetic attribution stress testing only.


In [8]:
out=RESULTS/'reviewer_closure'; out.mkdir(parents=True,exist_ok=True)
manifest={'run_all':RUN_ALL,'utility_sensitivity':'executed' if RUN_UTILITY_SENSITIVITY else 'skipped','constraint_frontier':'executed' if RUN_CONSTRAINT_FRONTIER else 'skipped','divergent_selector':'protocol_only_primary_tie_retained','semi_real':'skipped_no_audited_intervention_data','integrated_scaling':'skipped_no_integrated_8_10_operators','yaml_changed':False}
(out/'closure_manifest.json').write_text(json.dumps(manifest,indent=2)); print(json.dumps(manifest,indent=2))

{
  "run_all": true,
  "utility_sensitivity": "executed",
  "constraint_frontier": "executed",
  "divergent_selector": "protocol_only_primary_tie_retained",
  "semi_real": "skipped_no_audited_intervention_data",
  "integrated_scaling": "skipped_no_integrated_8_10_operators",
  "yaml_changed": false
}
